# Statistical Analysis: Production Memory Safe Results
Conducting hypothesis tests to see if differences in ID and log determinants between baselines are statistically significant.

In [ ]:
import pandas as pd
import glob
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from IPython.display import display

def load_all_jsonl(file_pattern):
    all_files = glob.glob(f'../results/production_memory_safe/**/{file_pattern}', recursive=True)
    dfs = []
    for file in all_files:
        model = file.split('/')[-3]
        df = pd.read_json(file, lines=True)
        df['model'] = model
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

id_df = load_all_jsonl('id.jsonl')
jacob_df = load_all_jsonl('jacobian.jsonl')

## ANOVA on Intrinsic Dimension (TwoNN)

In [ ]:
# Is there a significant difference in ID across baselines?
if not id_df.empty:
    id_twonn = id_df[id_df['estimator'] == 'twonn'].dropna(subset=['id_estimate'])
    if not id_twonn.empty:
        try:
            model = ols('id_estimate ~ C(baseline) + C(model) + depth', data=id_twonn).fit()
            anova_table = sm.stats.anova_lm(model, typ=2)
            display(anova_table)
        except Exception as e:
            print(f"ANOVA failed: {e}")

In [ ]:
# Post-hoc pairwise T-tests between trained and baselines
if not id_df.empty:
    id_twonn = id_df[id_df['estimator'] == 'twonn'].dropna(subset=['id_estimate'])
    for model_name in id_twonn['model'].unique():
        print(f"\nModel: {model_name} (TwoNN ID)")
        print("-"*50)
        m_df = id_twonn[id_twonn['model'] == model_name]
        trained_id = m_df[m_df['baseline'] == 'trained']['id_estimate']
        for baseline in m_df['baseline'].unique():
            if baseline != 'trained':
                other_id = m_df[m_df['baseline'] == baseline]['id_estimate']
                if len(trained_id) > 1 and len(other_id) > 1:
                    t_stat, p_val = stats.ttest_ind(trained_id, other_id, equal_var=False)
                    print(f"{baseline:30s} | t-stat: {t_stat:>8.3f} | p-value: {p_val:>8.2e}")

## Jacobian Deteriminant Stability

In [ ]:
# Checking if baselines significantly change the log abs determinant compared to trained
if not jacob_df.empty:
    for model_name in jacob_df['model'].unique():
        print(f"\nModel: {model_name} (Log Abs Det)")
        print("-" * 50)
        df_m = jacob_df[jacob_df['model'] == model_name].dropna(subset=['log_abs_det'])
        
        trained_det = df_m[df_m['baseline'] == 'trained']['log_abs_det']
        
        for baseline in df_m['baseline'].unique():
            if baseline != 'trained':
                other_det = df_m[df_m['baseline'] == baseline]['log_abs_det']
                if len(trained_det) > 1 and len(other_det) > 1:
                    t_stat, p_val = stats.ttest_ind(trained_det, other_det, equal_var=False)
                    print(f"{baseline:30s} | t-stat: {t_stat:>8.3f} | p-value: {p_val:>8.2e}")

## Summary of Condition Number Exceedances (Kappa Alerts)

In [ ]:
# Count of kappa alerts (condition numbers blowing up)
if not jacob_df.empty:
    alert_summary = jacob_df.groupby(['model', 'baseline'])['kappa_alert'].agg(['sum', 'count'])
    alert_summary['alert_rate_%'] = (alert_summary['sum'] / alert_summary['count']) * 100
    display(alert_summary)